# Õppetund 11 - Mudeli konteksti protokoll (MCP)

The **Mudeli konteksti protokoll (MCP)** on avatud standard, mis võimaldab agenditel dünaamiliselt avastada ja kasutada tööriistu, ressursse ja andmeallikaid käituse ajal. Selle asemel, et tööriistu agendisse kõvakodeerida, võimaldab MCP agentidel ühendada end välistesse serveritesse, mis pakuvad võimeid nõudmisel.

In this lesson, you'll learn:
- Mis on MCP ja miks see on oluline agentisüsteemide jaoks
- Kuidas MCP klient-serveri arhitektuur töötab
- Kuidas ehitada agente, kes kasutavad MCP-stiilis tööriistade avastamist


## Seadistamine

**Eeltingimused:**
- Azure AI Foundry projekt, kus on juurutatud mudel
- Käivitage `az login` autentimiseks


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Mis on Model Context Protocol (MCP)?

MCP määratleb standardse viisi, kuidas tehisintellekti agendid saavad avastada ja suhelda väliste tööriistade ning andmeallikatega:

- **MCP Server**: Avaldab tööriistu, ressursse ja prompte standardse protokolli kaudu
- **MCP Client**: Agendi käituskeskkond, mis ühendub serveritega ja avastab saadaolevad võimalused
- **Dünaamiline avastamine**: Agentidel ei ole vaja kõvakodeeritud tööriistu — nad avastavad, mis on käituse ajal saadaval

See on võimas viis laiendatavate agendisüsteemide ehitamiseks, kus uusi võimalusi saab lisada ilma agendi koodi muutmata.


## Kuidas MCP töötab

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Agent (MCP client) ühendub MCP serveriga
2. Server vastab saadaolevate tööriistade ja nende skeemide nimekirjaga
3. Agent võib seejärel oma arutluse käigus kutsuda ükskõik millist avastatud tööriista
4. Tulemused tagastatakse sama protokolli kaudu


## MCP-tööriistade avastamise simuleerimine

Kuna tõeline MCP-server nõuab töötavat serveriprotsessi, demonstreerime seda mustrit, kasutades `@tool` funktsioone, mis simuleerivad seda, mida MCP-ühendusega majutusteenus pakuks.

Tootmises avastatakse need tööriistad dünaamiliselt MCP-serverist, mitte määratletakse neid lokaalselt.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## MCP-stiilis tööriistadega agendi loomine


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP tootmiskeskkonnas

Tootmiskeskkonnas võimaldab MCP võimsaid mustreid:

- **Dünaamiline tööriistade avastamine**: Agentid ühenduvad MCP-serveritega ja avastavad tööriistu jooksvalt
- **Eraldatud arhitektuur**: Tööriistapakkujad saavad agentist sõltumatult uuendusi teha
- **Organisatsioonideülene jagamine**: Meeskonnad saavad MCP-serverite kaudu teha võimekusi kättesaadavaks, mida iga agent saab kasutada
- **Microsoft Agent Frameworki tugi**: MAF-il on sisseehitatud MCP-kliendi tugi läbi `mcp` integratsiooni

Reaalse MCP-serveri kasutamiseks MAF-iga ühendutakse läbi `hosted_mcp_tool()` või MCP-kliendi integratsiooni.

**Lisateave:**
- [MCP-spetsifikatsioon](https://modelcontextprotocol.io/)
- [Microsoft Agent Frameworki MCP tugi](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Kokkuvõte

Selles õppetükis õppisite:
- **MCP** on avatud standard agentide ja tööriistapakkujate vaheliseks tööriistade dünaamiliseks avastamiseks
- **klient-teenuse arhitektuur** võimaldab agentidel käitusajal võimekusi avastada
- MCP võimaldab **laiendatavaid, lahuselt seotud agendisüsteeme**, kuhu tööriistu saab lisada ilma koodimuudatusteta
- Microsoft Agent Framework pakub tootmiskasutuseks **sisseehitatud MCP-tuge**


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastutusest loobumine**:
See dokument on tõlgitud tehisintellekti tõlketeenuse [Co-op Translator](https://github.com/Azure/co-op-translator) abil. Kuigi me püüame tagada täpsust, palun arvestage, et automaatsed tõlked võivad sisaldada vigu või ebatäpsusi. Originaaldokument selle emakeeles tuleks pidada autoriteetseks allikaks. Olulise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta käesoleva tõlke kasutamisest tulenevate arusaamatuste ega valesti tõlgendamise eest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
